In [41]:
# Load env variables and create client
import os

from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

api_key = os.getenv("CLAUDE_API_KEY")
client = Anthropic(api_key=api_key)
model = "claude-haiku-4-5"

In [42]:
# Helper functions

from anthropic.types import Message

def add_user_message(messages, message):
    user_message = {"role": "user", "content": message.content if isinstance(message, Message) else message}
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {"role": "assistant", "content": message.content if isinstance(message, Message) else message}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[], tools=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }
    if tools: 
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message

def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

In [58]:
# Tools and Schemas

from datetime import datetime, timedelta
from anthropic.types import ToolParam

def add_duration_to_datetime(
    datetime_str, duration=0, unit="days", input_format="%Y-%m-%d"
):
    date = datetime.strptime(datetime_str, input_format)

    if unit == "seconds":
        new_date = date + timedelta(seconds=duration)
    elif unit == "minutes":
        new_date = date + timedelta(minutes=duration)
    elif unit == "hours":
        new_date = date + timedelta(hours=duration)
    elif unit == "days":
        new_date = date + timedelta(days=duration)
    elif unit == "weeks":
        new_date = date + timedelta(weeks=duration)
    elif unit == "months":
        month = date.month + duration
        year = date.year + month // 12
        month = month % 12
        if month == 0:
            month = 12
            year -= 1
        day = min(
            date.day,
            [
                31,
                29 if year % 4 == 0 and (year % 100 != 0 or year % 400 == 0) else 28,
                31,
                30,
                31,
                30,
                31,
                31,
                30,
                31,
                30,
                31,
            ][month - 1],
        )
        new_date = date.replace(year=year, month=month, day=day)
    elif unit == "years":
        new_date = date.replace(year=date.year + duration)
    else:
        raise ValueError(f"Unsupported time unit: {unit}")

    return new_date.strftime("%A, %B %d, %Y %I:%M:%S %p")


def set_reminder(content, timestamp):
    print(f"----\nSetting the following reminder for {timestamp}:\n{content}\n----")


add_duration_to_datetime_schema = ToolParam({
    "name": "add_duration_to_datetime",
    "description": "Adds a specified duration to a datetime string and returns the resulting datetime in a detailed format. This tool converts an input datetime string to a Python datetime object, adds the specified duration in the requested unit, and returns a formatted string of the resulting datetime. It handles various time units including seconds, minutes, hours, days, weeks, months, and years, with special handling for month and year calculations to account for varying month lengths and leap years. The output is always returned in a detailed format that includes the day of the week, month name, day, year, and time with AM/PM indicator (e.g., 'Thursday, April 03, 2025 10:30:00 AM').",
    "input_schema": {
        "type": "object",
        "properties": {
            "datetime_str": {
                "type": "string",
                "description": "The input datetime string to which the duration will be added. This should be formatted according to the input_format parameter.",
            },
            "duration": {
                "type": "number",
                "description": "The amount of time to add to the datetime. Can be positive (for future dates) or negative (for past dates). Defaults to 0.",
            },
            "unit": {
                "type": "string",
                "description": "The unit of time for the duration. Must be one of: 'seconds', 'minutes', 'hours', 'days', 'weeks', 'months', or 'years'. Defaults to 'days'.",
            },
            "input_format": {
                "type": "string",
                "description": "The format string for parsing the input datetime_str, using Python's strptime format codes. For example, '%Y-%m-%d' for ISO format dates like '2025-04-03'. Defaults to '%Y-%m-%d'.",
            },
        },
        "required": ["datetime_str"],
    },
})

set_reminder_schema = ToolParam({
    "name": "set_reminder",
    "description": "Creates a timed reminder that will notify the user at the specified time with the provided content. This tool schedules a notification to be delivered to the user at the exact timestamp provided. It should be used when a user wants to be reminded about something specific at a future point in time. The reminder system will store the content and timestamp, then trigger a notification through the user's preferred notification channels (mobile alerts, email, etc.) when the specified time arrives. Reminders are persisted even if the application is closed or the device is restarted. Users can rely on this function for important time-sensitive notifications such as meetings, tasks, medication schedules, or any other time-bound activities.",
    "input_schema": {
        "type": "object",
        "properties": {
            "content": {
                "type": "string",
                "description": "The message text that will be displayed in the reminder notification. This should contain the specific information the user wants to be reminded about, such as 'Take medication', 'Join video call with team', or 'Pay utility bills'.",
            },
            "timestamp": {
                "type": "string",
                "description": "The exact date and time when the reminder should be triggered, formatted as an ISO 8601 timestamp (YYYY-MM-DDTHH:MM:SS) or a Unix timestamp. The system handles all timezone processing internally, ensuring reminders are triggered at the correct time regardless of where the user is located. Users can simply specify the desired time without worrying about timezone configurations.",
            },
        },
        "required": ["content", "timestamp"],
    },
})

batch_tool_schema = ToolParam({
    "name": "batch_tool",
    "description": "Invoke multiple other tool calls simultaneously",
    "input_schema": {
        "type": "object",
        "properties": {
            "invocations": {
                "type": "array",
                "description": "The tool calls to invoke",
                "items": {
                    "type": "object",
                    "properties": {
                        "name": {
                            "type": "string",
                            "description": "The name of the tool to invoke",
                        },
                        "arguments": {
                            "type": "string",
                            "description": "The arguments to the tool, encoded as a JSON string",
                        },
                    },
                    "required": ["name", "arguments"],
                },
            }
        },
        "required": ["invocations"],
    },
})

pass

In [44]:
from anthropic.types import ToolParam
def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    """
    Returns the current date and time formatted according to the provided date_format string. The date_format parameter should follow Python's strftime format codes, allowing for flexible output formats. For example, using '%Y-%m-%d %H:%M:%S' will return a string like '2025-04-03 14:30:00', while '%A, %B %d, %Y' will return 'Thursday, April 03, 2025'. This tool is useful for obtaining the current datetime in various formats for display, logging, or further processing in applications that require time-sensitive information.
    """
    if not date_format:
        raise ValueError("date_format cannot be empty")
    now = datetime.now()
    return now.strftime(date_format)


get_current_datetime_schema  = ToolParam({
  "name": "get_current_datetime",
  "description": "Returns the current date and time formatted according to the provided format string. Useful for obtaining the current datetime in various formats for display, logging, or time-sensitive processing.",
  "input_schema": {
    "type": "object",
    "properties": {
      "date_format": {
        "type": "string",
        "description": "A Python strftime format string that controls the output format. For example: '%Y-%m-%d %H:%M:%S' returns '2025-04-03 14:30:00', '%A, %B %d, %Y' returns 'Thursday, April 03, 2025'. Must not be empty."
      }
    },
    "required": []
  }
})



In [59]:
messages = []

messages.append({
    "role": "user",
    "content": "what is the exact time formatted as 'HH:mm:ss'?"
})

response = client.messages.create(
    model=model,
    messages=messages,
    max_tokens=1000,
    tools=[get_current_datetime_schema, set_reminder_schema, add_duration_to_datetime_schema],
)

messages.append({
    "role": "assistant",
    "content": response.content  
})

messages


[{'role': 'user',
  'content': "what is the exact time formatted as 'HH:mm:ss'?"},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_01KKPQfUJgit4giHw3nSdYLS', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use')]}]

In [46]:
exact_time = get_current_datetime(**response.content[0].input)

In [47]:
messages.append({
    "role": 'user',
    "content": [
        {
            "type": "tool_result",
            "tool_use_id": response.content[0].id,
            "content": exact_time,
            "is_error": False
        }
    ],

})
messages


[{'role': 'user',
  'content': "what is the exact time formatted as 'HH:mm:ss'?"},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_0111Lnf7crEd3eD3fMqecr24', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use')]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_0111Lnf7crEd3eD3fMqecr24',
    'content': '17:35:39',
    'is_error': False}]}]

In [60]:
messages = []

add_user_message(messages, "what is the exact time formatted as 'HH:mm:ss'?")

response = client.messages.create(
    model=model,
    messages=messages,
    max_tokens=1000,
    tools=[get_current_datetime_schema, set_reminder_schema, add_duration_to_datetime_schema]
)

add_assistant_message(messages, response.content)

messages



[{'role': 'user',
  'content': "what is the exact time formatted as 'HH:mm:ss'?"},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_01WLeaLTGcMdYDyGGzcVun8y', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use')]}]

In [61]:

import json

def run_tool(tool_name, tool_input ):
    if tool_name == 'get_current_datetime':
        return get_current_datetime(**tool_input)
    elif tool_name == 'add_duration_to_datetime':
        return add_duration_to_datetime(**tool_input)
    elif tool_name == 'set_reminder':
        return set_reminder(**tool_input)
      

def run_tools(message):
    tool_reqs = [block for block in message.content if block.type == "tool_use"]
    tool_results = []

    for tool_req in tool_reqs:
        try:
            tool_output =run_tool(tool_req.name, tool_req.input)
            tool_results.append({
                "type": "tool_result",  
                "tool_use_id": tool_req.id,
                "content": json.dumps(tool_output),
                "is_error": False
            })
        except Exception as e:
            tool_results.append({
                "type": "tool_result",
                "tool_use_id": tool_req.id,
                "content": f"Error: {str(e)}",
                "is_error": True
            })
    return tool_results


In [63]:
def run_conversation(messages):
    while True:
        response = chat(messages, tools=[get_current_datetime_schema, add_duration_to_datetime_schema, set_reminder_schema])
        add_assistant_message(messages, response)
        print(text_from_message(response))
        if response.stop_reason != "tool_use":
            break
        tool_results = run_tools(response)

        add_user_message(messages, tool_results)
    return messages

In [64]:
messages = []
add_user_message(messages, "set a reminder for me to meet my dentist in 100 days from now")
run_conversation(messages)

I'll set a reminder for you to meet your dentist in 100 days from now. First, let me get the current date and time, then calculate the date 100 days from now.
Now let me add 100 days to this date:
Perfect! Now let me set the reminder:
----
Setting the following reminder for 2026-08-17T17:55:48:
Meet your dentist
----
Done! I've set a reminder for you to meet your dentist on **Monday, August 17, 2026 at 5:55 PM** (100 days from now).


[{'role': 'user',
  'content': 'set a reminder for me to meet my dentist in 100 days from now'},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text="I'll set a reminder for you to meet your dentist in 100 days from now. First, let me get the current date and time, then calculate the date 100 days from now.", type='text'),
   ToolUseBlock(id='toolu_01Cc3r65ww78Ut8jfYbw3pph', caller=DirectCaller(type='direct'), input={'date_format': '%Y-%m-%d %H:%M:%S'}, name='get_current_datetime', type='tool_use')]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_01Cc3r65ww78Ut8jfYbw3pph',
    'content': '"2026-05-09 17:55:48"',
    'is_error': False}]},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text='Now let me add 100 days to this date:', type='text'),
   ToolUseBlock(id='toolu_01GmUyBNaG1ZkbNMf6ux8y3q', caller=DirectCaller(type='direct'), input={'datetime_str': '2026-05-09 17:55:48', 'duration': 100, 'unit': 'days', 'input_form